In [2]:
import pandas as pd
import numpy as np

# load dataset
df = pd.read_csv("../data/cleaned_dataset/cleaned_ticket_prices.csv")

df.head()


,month,conflict_phase,airline,iata_code,country,region,airline_type,route_class,avg_route_km,base_fare_usd,...,yoy_surcharge_change_pct,year,month_num,quarter,is_extreme_fare,fuel_surcharge_ratio,taxes_ratio,base_ratio,crude_jet_ratio,fare_per_km
0,2019-01,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1179.91,...,0.0,2019,1,2019Q1,False,0.0731,0.1171,0.8098,1.1838,0.1714
1,2019-02,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1176.08,...,0.0,2019,2,2019Q1,False,0.0468,0.0933,0.8599,1.2087,0.1609
2,2019-03,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1133.88,...,0.0,2019,3,2019Q1,False,0.0857,0.0876,0.8267,1.1672,0.1614
3,2019-04,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1237.95,...,0.0,2019,4,2019Q2,False,0.0402,0.1155,0.8442,1.1320,0.1725
4,2019-05,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1270.08,...,0.0,2019,5,2019Q2,False,0.0570,0.1119,0.8311,1.1913,0.1798


KPI Calculations

KPI 1 — Price Pass-through (Fuel → Fare impact)

In [6]:
df['price_pass_through'] = df['total_fare_usd'] / df['jet_fuel_usd_barrel']
df[['total_fare_usd','jet_fuel_usd_barrel','price_pass_through']].head()

,total_fare_usd,jet_fuel_usd_barrel,price_pass_through
0,1457.07,74.58,19.537007
1,1367.65,81.72,16.735805
2,1371.61,76.87,17.843242
3,1466.34,73.34,19.993728
4,1528.12,72.97,20.941757


Insight

Fuel price increases are only partially reflected in ticket prices, showing delayed cost transfer. During conflict phases, pass-through becomes stronger, indicating reactive pricing behavior.

KPI 2 — Fuel Shock Indicator

In [7]:
df['fuel_price_change_pct'] = df.groupby('airline')['jet_fuel_usd_barrel'].pct_change() * 100
df['fuel_shock_flag'] = df['fuel_price_change_pct'] > 10

df[['jet_fuel_usd_barrel','fuel_price_change_pct','fuel_shock_flag']].head(10)

,jet_fuel_usd_barrel,fuel_price_change_pct,fuel_shock_flag
0,74.58,NaN,False
1,81.72,9.573612,False
2,76.87,-5.934900,False
3,73.34,-4.592169,False
4,72.97,-0.504500,False
5,73.26,0.397424,False
6,71.11,-2.934753,False
7,80.12,12.670510,True
8,75.79,-5.404393,False
9,76.97,1.556934,False


Significant fuel shocks align with geopolitical events. However, not all shocks result in proportional fare increases, indicating strategic pricing control by airlines.

KPI 3 — Fare Volatility

In [8]:
df['fare_volatility'] = df.groupby('route_class')['total_fare_usd'].transform('std')
df[['route_class','fare_volatility']].drop_duplicates()

,route_class,fare_volatility
0,Long-Haul,655.430230
87,Medium-Haul,214.154519
174,Short-Haul Domestic,36.233239
261,Short-Haul Regional,79.230064
348,Ultra-Long-Haul,1545.403435


In [ ]:
Long-haul routes show significantly higher volatility, making them more sensitive to global disruptions like fuel price shocks.

KPI 4 — Revenue Proxy

In [9]:
df['revenue_proxy'] = df['total_fare_usd'] * df['load_factor_pct']
df[['total_fare_usd','load_factor_pct','revenue_proxy']].head()

,total_fare_usd,load_factor_pct,revenue_proxy
0,1457.07,79.4,115691.358
1,1367.65,89.0,121720.850
2,1371.61,64.1,87920.201
3,1466.34,75.1,110122.134
4,1528.12,85.8,131112.696


Insight:
Revenue depends heavily on load factor + fare together, not just pricing. Even with lower fares, high demand can drive revenue growth.

KPI 5 — Surcharge Coverage Ratio